In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("SaltingDemo").getOrCreate()

In [0]:
display(spark)

In [0]:
# Orders Table — customer 1 is heavily skewed (8 out of 12 rows)
orders_data = [
    (1, "C001", 100.0),
    (2, "C001", 200.0),
    (3, "C001", 150.0),
    (4, "C001", 300.0),
    (5, "C001", 250.0),
    (6, "C001", 180.0),
    (7, "C001", 220.0),
    (8, "C001", 190.0),   # C001 has 8 rows — SKEWED
    (9, "C002", 500.0),   # C002 has 2 rows
    (10,"C002", 400.0),
    (11,"C003", 700.0),   # C003 has 1 row
    (12,"C004", 350.0),   # C004 has 1 row
]

# Customers Table — small reference table
customers_data = [
    ("C001", "Alice",   "New York"),
    ("C002", "Bob",     "London"),
    ("C003", "Charlie", "Singapore"),
    ("C004", "Diana",   "Dubai"),
]

df_orders    = spark.createDataFrame(orders_data,    ["order_id", "customer_id", "amount"])
df_customers = spark.createDataFrame(customers_data, ["customer_id", "name", "city"])

print("===== ORDERS TABLE =====")
df_orders.show()

print("===== CUSTOMERS TABLE =====")
df_customers.show()


In [0]:
print("===== SKEW CHECK — ROW DISTRIBUTION =====")
df_orders.groupBy("customer_id") \
         .count() \
         .orderBy(F.desc("count")) \
         .show()

In [0]:
print("===== JOIN WITHOUT SALTING =====")
df_normal_join = df_orders.join(df_customers, on="customer_id", how="inner")
df_normal_join.show()

In [0]:
df_normal_join.groupBy(F.spark_partition_id()) \
              .count() \
              .orderBy(F.spark_partition_id()) \
              .show()

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
print("===== APPLYING SALTING =====")

salt_factor = 4  

df_orders_salted = df_orders \
    .withColumn("salt",(F.rand(seed=42) * salt_factor).cast("int") ) \
    .withColumn("salted_key",F.concat(col("customer_id"), F.lit("_"), col("salt").cast("string"))
    )

print("Orders after salting:")
df_orders_salted.show()


In [0]:
# ----- SMALL TABLE (Customers) -----
# Explode to create 4 copies of each customer row (one per salt value)
df_customers_exploded = df_customers \
    .withColumn("salt_array", F.array([F.lit(i) for i in range(salt_factor)])) \
    .withColumn("salt", F.explode("salt_array")) \
    .withColumn( "salted_key",F.concat(col("customer_id"), F.lit("_"), col("salt").cast("string")) ) \
    #.drop("salt_array", "salt")

print("Customers after exploding:")
df_customers_exploded.show()

In [0]:
# Join on salted key — skew is now resolved
df_salted_join = df_orders_salted \
    .join(df_customers_exploded, on="salted_key", how="inner") \
    .drop("salt", "salted_key", "salt_array")

print("===== FINAL RESULT AFTER SALTING =====")
df_salted_join.show()

In [0]:

# Check partition distribution — should be more even now
print("Partition sizes after salted join:")
df_salted_join.groupBy(F.spark_partition_id()) \
              .count() \
              .orderBy(F.spark_partition_id()) \
              .show()